# Precision Agriculture Analysis Pipeline

This notebook acts as an end-to-end execution pipeline utilizing the `Precision Agriculture Spatial Toolkit`.
It sequentially processes raw shapefiles into a regularized spatial grid, models yield outcomes using spatial cross-validation, and interprets the results using SHAP and PDPs.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
import os

sys.path.append('../scripts')


# 1. Data IO
from data_imports import files_to_dict, cast_to_crs, load_schema, rename_columns_fuzzy
# 2. Preprocessing
from preprocessing import trial_boundaries, remove_boundary_data, filter_parameter, remove_local_outliers, remove_spatial_outliers, text_encoder
# 3. Sensor Delay Correction
from delay import run_robust_pcdi, manual_lag_correction, prepare_shifted_data
# 4. Topography
from terrain import terrain
# 5. Spatial Aggregation
from aggregation import calculate_dominant_heading, create_spatial_grid, aggregate_points_to_grid, assign_polygon_to_grid, align_to_grid, add_spatial_lag, kmeans_clustering, impute_spatial_mean
# 6. Spatial Modeling
from modelling import determine_optimal_block_size, spatial_checkerboard_split, plot_checkerboard_split, rf_predict_spatial, xgb_predict_spatial, compare_spatial_models, tune_spatial_optuna, train_final_model
# 7. Interpretability
from interpretation import run_full_diagnostics, shap_values, plot_three_pdp_ice, two_dim_pdp_plots

%matplotlib inline

## 1. Data Imports & Standardization
Load raw shapefiles from a target directory, align their columns to a standard schema, and assign a uniform Coordinate Reference System (CRS).

In [ ]:
# NOTE: Replace 'data_folder' with the actual path to your shapefiles directory

dir_path = 'dir_path_here'
import_dict, file_names = files_to_dict(dir_path)


# cast 
crs = 'EPSG:32734' # can also leave this as none, this is the default
working_dict = cast_to_crs(import_dict, crs)

# Load standardization schema for column mapping
schema = load_schema('../scripts/schema_mapping.json')

# Standardize column names using rapidfuzz string matching
for key in working_dict:
    working_dict[key] = rename_columns_fuzzy(working_dict[key], schema, threshold=80, exclude_cols=['geometry'])


In [ ]:
yield_colname = 'yield'  # Replace with the actual column name for yield in your data
seed_colname = 'seeding_rate'  # Replace with the actual column name for seeding rate in your data
target_seed = 'trial_seeding_rate'  # Replace with the actual column name for seeding rate in your data
target_fert = 'trial_fertilizer_rate'  # Replace with the actual column name for fertilizer rate in your data

# Extract target dataframes for pipeline demonstration 
# (Replace the dictionary keys with your actual file names)
working_df   = working_dict[yield_colname]                    # raw yield map
input_df     = working_dict[seed_colname]  # seeder input map
trial_df_s   = working_dict[target_seed]      # trial design polygons
trial_df_n   = working_dict[target_fert]      # trial design polygons
soil_df      = working_dict['Bosplaas_soil_poly']       # soil polygons
boundary_df  = working_dict['Canola Kamp 1']            # boundary polygon

## 2. Preprocessing & Cleaning
Filter out non-trial areas, headlands, statistical outliers, and spatial anomalies to ensure the model trains on high-quality agronomic data.

In [ ]:
# boundary enforcement on yield map
# should be "outer" boundary of field at this point, not trial design
# working_df: the df on which the field boundary will be enforced,
# trial_df: the df containing the trial design polygons
# optional to plot the results, and specify the column to use in the plots (default false)
yield_df_inside, removed_yield_data = remove_boundary_data(
                                        working_df,
                                        trial_df_s,
                                        buffer_zone=20,
                                        plots=True,
                                        colname=yield_colname)

In [ ]:
# field boundary enforcement on recorded input maps (assuming no boundary file)
# should be "outer" boundary of field at this point, not trial design
# input_df: the df on which the field boundary will be enforced,
# trial_df: the df containing the trial design polygons
# optional to plot the results, and specify the column to use in the plots (default false)
input_df_inside, removed_input_data = remove_boundary_data(
                                        input_df,
                                        trial_df_s,
                                        buffer_zone=20,
                                        plots=True,
                                        colname=seed_colname)

In [ ]:
# terrain data
crit = 'dist' # dist for radius around point, neighbours for no of nearest neighbours
s = 30 # radius in meters or number of nearest neighbours to consider
terrain_df = terrain(
    x = working_df.geometry.x,
    y = working_df.geometry.y,
    z = working_df['elevation'],
    crit='dist',
    s=s
)

# Convert degrees to radians first
aspect_rad = np.radians(terrain_df['aspect_deg'])

terrain_df['aspect_north'] = (np.cos(aspect_rad))
terrain_df['aspect_east']  = (np.sin(aspect_rad))

In [ ]:
# 1HE of soil columns
# returns df with encoded columns and list of new column names to include in prediction
# Select columns with 'object' or 'category' dtypes
cat_cols = soil_df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols.pop(cat_cols.index('Gat_nr_'))
print("Categorical columns in soil_df:", cat_cols)

# return encoded df
encoded_df, soil_colnames = text_encoder(cat_cols, soil_df)

In [ ]:
# join everything together
# don't join elevation col from terrain DF - it will be a duplicate
yield_df_inside = pd.concat([yield_df_inside, terrain_df[['slope_deg','aspect_north','aspect_east']]], axis=1)
yield_df_inside = gpd.sjoin(yield_df_inside, encoded_df, how='left', predicate='intersects').drop(columns='index_right')

3) Input cleaning
   1) flow delay if needed
   2) speed, turning points and global outliers
   3) Trial boundaries (optional)

In [ ]:
# Convert seeding rate from t/ha to kg/ha for consistency - not always necessary, but good practice to ensure consistency across datasets
input_df['applied_rate'] = input_df['applied_rate']*1000 # convert from t/ha to kg/ha

In [ ]:
# speed and global outlier removal from recorded input maps
# NB these filters should be very broad, so as to only remove extreme outliers

# below line only if flow correction isn't done
time_corrected_input_df = input_df_inside.copy() # replace with actual time-corrected df after flow delay correction

# speed filter
seed_colname = 'applied_rate' # column name for seed flow rate
parameter_colname = 'speed_km_h' # column name for the parameter of interest
# removes all parameter points below 10th and above 90th percentile by default
# upper_threshold and lower_threshold can be set to specific values if desired
# plots true by default to visualize the parameter distribution but can be set to false
# and thresholds on a histogram after cleaning
# Function can be used for any parameter that I want to remove outliers from
# by changing the parameter_colname and seed_colname inputs
time_corrected_input_df = filter_parameter(
    time_corrected_input_df,
    seed_colname,
    parameter_colname,
    'percentile',
    lower_threshold=0.05,
    upper_threshold=0.95,
    plots=True
)

In [ ]:
# global seeding rate outlier removal
input_colname = 'target_seed' # column name for seed rate
seed_min = trial_df_s[input_colname].min()*0 # set to half of minimum seed rate in trial design, can be adjusted as needed
seed_max = trial_df_s[input_colname].max()*1.5 # set to one and a half times the maximum seed rate in trial design, can be adjusted as needed
filtered_input_df = filter_parameter(time_corrected_input_df, seed_colname, seed_colname,'bounds', seed_min, seed_max)


In [ ]:
# trial design boundary enforcement on recorded input maps (assuming no boundary file)
# trial_df: the df containing the full trial design
# optional to plot the results, and specify the column to use in the plots (default false)
trial_polygons = trial_boundaries(
    trial_df_s,
    plots= False,
    colname='target_seed'
)

In [ ]:
# trial design boundary enforcement (optional)
# input_df: the cleaned input df on which the trial design boundary will be enforced,
# trial_polygons: the df containing only the trial design polygons
# optional to plot the results, and specify the column to use in the plots (default false)
trial_bound_input_df, trial_bound_removed_input_data = remove_boundary_data(
    filtered_input_df,
    trial_polygons,
    plots=False,
    colname='applied_rate'
)

In [ ]:
# attempt at removing spatial outliers
trial_bound_input_df = remove_spatial_outliers(
                        gdf = trial_bound_input_df,
                        dist = 15,
                        n = 5,
                        plots = True,
                        value_col = 'applied_rate',
                        max_iters= 10
)

4) Yield map cleaning
   1) header status
   2) flow delay
   3) speed
      1) high & low
      2) high ROC
   4) trial boundaries
   5) remaining turning points, short passes and narrow swathes
   6) outliers (global)
      1) biological limits
      2) SD - vary level based on if local cleaning will be done or not
   7) Outliers (local - optional)
      1) uses neighbourhood of points and calculates SD based off that

In [ ]:
augmented_df = yield_df_inside[(yield_df_inside[yield_colname].notna())]

# trial design boundary enforcement (optional)
# yield_df: the cleaned yield df on which the trial design boundary will be enforced,
# trial_polygons: the df containing only the trial design polygons
# optional to plot the results, and specify the column to use in the plots (default false)
trial_bound_yield_df, trial_bound_removed_yield_data = fun.remove_boundary_data(
    augmented_df,
    trial_polygons,
    plots=False,
    colname=yield_colname
)


In [ ]:
working_df = working_df[(working_df[yield_colname] <10) ]
yield_df_inside = yield_df_inside[(yield_df_inside[yield_colname] <10) ]
trial_bound_yield_df = trial_bound_yield_df[(trial_bound_yield_df[yield_colname] <10) ]



In [ ]:
# header status
yield_colname = 'yield_mass_dry' # column name for yield
augmented_df = trial_bound_yield_df[(trial_bound_yield_df[yield_colname] > 0) & (trial_bound_yield_df[yield_colname].notna())]
# augmented_df_soil = yield_df_inside_soil[(yield_df_inside_soil[yield_colname] > 0) & (yield_df_inside_soil[yield_colname].notna())]

In [ ]:
# speed outlier removal from recorded yield maps
speed_colname = 'speed_km_h' # column name for the parameter of interest
filtered_yield_df = filter_parameter(
    augmented_df,
    yield_colname,
    speed_colname,
    method = 'SD',
    lower_threshold=3,
    upper_threshold=3,
    plots=True
)

In [ ]:
# remaining turning points, short passes and narrow swathes
final_yield_df = gpd.geodataframe
# outliers (global)
# biological limits
final_yield_df = filter_parameter(
    trial_bound_yield_df,
    yield_colname,
    yield_colname,
    method = 'bounds',
    lower_threshold= 0.0,
    upper_threshold= 5.0,
    plots=True
)


In [ ]:
# SD - vary level based on if local cleaning will be done or not
final_yield_df = fun.filter_parameter(
    final_yield_df,
    yield_colname,
    yield_colname,
    method = 'SD',
    lower_threshold=5,
    upper_threshold=5, # upper yield limit in t/ha
    plots=True
)

In [ ]:
# local outliers - this looks at k NN atm, want to implement 2 alternative options as well.
# Both a physical radius based calculation, as well as a way of only looking at points within
# the same pass so that spatial variation due to varying inputs don't lead to points being mistakenly identified as outliers
local_cleaned_yield_df, local_removed_yield_df = fun.remove_local_outliers(
    gdf = final_yield_df,
    value_col = yield_colname,
    crit = 'dist',
    s = 20,
    sd_threshold=3.0,
    plot_removed =True,
    plot_example = True
)

5) Join all data to our created grid (yield, inputs, soil, terrain)
   1) for points, take mean of all points in a block
   2) for polygons either assign or use area weighted contributions


In [ ]:
vmax = local_cleaned_yield_df[yield_colname].max()
vmin = local_cleaned_yield_df[yield_colname].min()
ticks = ticker.MaxNLocator(nbins=8).tick_values(vmin, vmax)


In [ ]:
#final_yield_df = final_yield_df.drop(columns = ['Date_left','Date_right','timestamp_left','timestamp_right'])
trial_bound_input_df = trial_bound_input_df.drop(columns = ['timestamp','Date'])

In [ ]:
final_yield_df = local_cleaned_yield_df.copy()
final_yield_df = final_yield_df.drop(columns = ['Date','timestamp'])

In [ ]:
# grid creation (with rotation)
# calculate rotation angle based on primary harvester heading
rotation_angle = fun.calculate_dominant_heading(
    points_df = final_yield_df,
    plots = True
    ) 

# create blank grid, squares, rectangles, rotated or N-E aligned
rotated_blank_grid, mean_swath_width = fun.create_spatial_grid(
    input_df = final_yield_df,
    cell_size=5,
    width_col= 'swath_width',
    length_col= 'distance_m',
    shape = 'rectangle',
    angle = rotation_angle,
    # the following three only needed for plotting, defaults to false and none
    plots = True,
    value_col = 'heading_deg',
    boundary_df = boundary_df,
    buffer_radius = 0
)

In [ ]:
# columns to include onto the master grid. I have to be able to make this more elegant?
colnames = ['elevation','slope_deg', 'aspect_deg', 'soil_depth',
       'Oorgangsdi', 'soil_clay_a', 'Volume___G', 'soil_clay_b', 'Volume___1',
       'soil_clay_c', 'Volume___2', 'soil_moisture', 'PBWK_0_300',
       'PBWK_300_6', 'PBWK_effek', 'PBWK_eff_1', 'soil_form_Cv2100',
       'soil_form_Gs1111', 'soil_form_Gs2111', 'soil_form_Gs2211',
       'soil_form_Gs2212', 'soil_form_Oa1120', 'soil_texture_Leem',
       'soil_texture_Sandleem', 'Oorgangs_1_600', 'Oorgangs_1_800',
       'Oorgangs_1_Nee', 'Tekstuur_1_Leem', 'Tekstuur_1_Sandleem',
       'Struktuu_1_Swak_ontwikkel', 'Tekstuur_2_Klei',
       'Struktuu_2_Struktuurloos', 'Struktuu_2_Swak_ontwikkel',
       'Teksturele_U6_Oorwegend_lemerig_of_poreuse_slik',
       'Teksturele_U7_Oorwegend_kleierig_of_digte_slik',
       'Teksturele_U8_Sand_bo_en_klei_onder', 'Karbonaa_1_Nee',
       'Karbonaa_D_Nee', 'Keerlaag_T_Klipfraksie', 'Keerlaag_T_Saproliet',
       'Keerlaag_D_300', 'Keerlaag_D_500', 'Keerlaag_D_600', 'Keerlaag_D_800',
       'Denitrifik_Nee', 'Beraamde_I_Matig', 'Beraamde_I_Matig-vinnig',
       'Grondvorm1_Glenrosa', 'Grondvorm1_Oakleaf','x','y','aspect_north','aspect_east',
       'soil_form_Gs1211', 'soil_form_Gs2112', 'soil_form_Oa2110',
       'Tekstuur_1_Kleileem',
       'Karbonaa_D_900','kompak_beg_Nee', 'kompak_end_Nee'
       'Keerlaag_D_400','Beraamde_I_Matig-stadig']

In [ ]:
# Aggregate all data (point and polygon) to blank grid
# master_grid now has a 'Yield' column
master_grid = fun.aggregate_points_to_grid(
    points_df = final_yield_df, 
    grid_gdf = rotated_blank_grid, 
    value_col = 'yield_mass_dry'
)

for col in colnames:
    if col in final_yield_df.columns:
        master_grid = fun.aggregate_points_to_grid(
            points_df = final_yield_df,
            grid_gdf = master_grid,
            value_col = col
        )

master_grid = fun.aggregate_points_to_grid(
    points_df = trial_bound_input_df,
    grid_gdf = master_grid,
    value_col = 'applied_rate'
)

master_grid = fun.assign_polygon_to_grid(
    polygon_df = trial_df_s,
    grid_gdf = master_grid,
    value_col = 'target_seed'
)
master_grid = fun.assign_polygon_to_grid(
    polygon_df = trial_df_n,
    grid_gdf = master_grid,
    value_col = 'target_fert'
)

In [ ]:
NON_FEATURE_COLS = ['geometry', yield_colname, 'grid_id','gg_ht','gg_nv', 'field_name','field_name_left',
                    'Dataset_left', 'crop_type_left', 'feature_id_left', 'timestamp_left',
                    'duration_s_left', 'area_left', 'swath_width_left', 'y_offset_left',
                    'speed_km_h_left', 'distance_m_left', 'fuel_consumed', 'crop_flow_mass',
                    'moisture', 'pass_number_left', 'crop_flow_vol','yield_mass_wet',
                    'Yld_Mass_W', 'yield_vol_wet', 'yield_vol_dry', 'Prod_ha_h__left',
                    'Date_left', 'Gat_nr_', 'field_name_right', 'Dataset_right',
                    'crop_type_right', 'feature_id_right', 'area_right', 'swath_width_right',
                    'x_offset_left', 'z_offset_left', 'applied_rate_left', 'mass_flow_left',
                    'target_rate_left', 'y_offset_right', 'speed_km_h_right', 'distance_m_right',
                    'duration_s_right', 'heading_deg_right', 'timestamp_right', 'elevation_right',
                    'pass_number_right', 'Prod_ha_h__right', 'Prdt_Amt_left', 'Date_right',
                    'dist', 'Prdt_Amt','field_name', 'Dataset', 'crop_type', 'feature_id', 'area',
                    'swath_width','x_offset_right', 'z_offset_right', 'applied_rate_right', 'mass_flow_right',
                    'target_rate_right', 'y_offset', 'speed_km_h', 'distance_m', 'duration_s',
                    'heading_deg', 'timestamp', 'pass_number', 'Prod_ha_h_',
                    'Prdt_Amt_right', 'Date','x_offset', 'z_offset','mass_flow'
]

In [ ]:
pred_cols = [
    col for col in master_grid.columns
    if col not in NON_FEATURE_COLS
]
print(f"Predictor columns ({len(pred_cols)}): {pred_cols}")


In [ ]:
master_grid.reset_index(drop=True, inplace=True)  # Reset index to ensure it's sequential
imputed_df = master_grid.copy()  # Create a copy to store imputed values
yield_colname = 'yield_mass_dry'  # Column name for yield
impute_cols = pred_cols

for col in master_grid.columns:
    if col in impute_cols:
        print(f"Imputing column: {col}")
        imputed_col = ism(
            gdf=master_grid,
            column_name=col,
            method='knn',  # or 'knn', 'kriging', etc.
            k=25,  # number of neighbors for KNN
        )
        imputed_df[col] = imputed_col

In [ ]:
# Drop grid cells with NaN in any predictor or the target
# Corner cells or cells with no yield / input points will be NaN.
n_before = len(master_grid)
master_grid_clean = master_grid.dropna(subset=pred_cols + [yield_colname])
print(f"Grid cells after NaN drop: {len(master_grid_clean)} / {n_before} "
      f"({len(master_grid_clean)/n_before*100:.1f}%)")

# Drop grid cells with NaN in any predictor or the target
# Corner cells or cells with no yield / input points will be NaN.
n_before = len(imputed_df)
imputed_grid_clean = imputed_df.dropna(subset=pred_cols + [yield_colname])
print(f"Grid cells after NaN drop: {len(imputed_grid_clean)} / {n_before} "
      f"({len(imputed_grid_clean)/n_before*100:.1f}%)")

In [ ]:
# 1. Ask the field for its optimal block size
dynamic_block_size = fun.determine_optimal_block_size(
    df=imputed_grid_clean, 
    target_col='yield_mass_dry',
    harvester_width=mean_swath_width
)


In [ ]:
#block_size = dynamic_block_size  # Use the optimal block size determined above
block_size = 68 #mean_swath_width
# 2. Split the data using that optimal size
X_train, X_test, y_train, y_test, train_df, test_df = fun.spatial_checkerboard_split(
    df=imputed_grid_clean,
    pred_params=pred_cols,
    target_col='yield_mass_dry',
    block_size=8*mean_swath_width, 
    test_size=0.20,
    random_state = 69 # 44 has got test perf # 40 for better image # 27 feasible # 32 feasible # 39 feasible # 46 # 54 # 68 b # 69 b so far # 75 #80 pretty good # 102
)

# 3. Plot the result to ensure your test blocks cover the whole field
fun.plot_checkerboard_split(train_df, test_df)

In [ ]:
print(pred_cols)
trained_rf_model, trained_rf_model_features = fun.rf_predict_spatial(
                input_df = train_df,
                pred_params = pred_cols,
                target_col = 'yield_mass_dry',
                block_size= mean_swath_width
)

In [ ]:
model_analysis.run_full_diagnostics(trained_rf_model, X_train, y_train, X_test, y_test, pred_cols,'applied_rate','target_fert', metrics_only=True)

In [ ]:
from two_dim_pdp_plots import two_dim_pdp_plots

In [ ]:
feat_combos = np.array([['applied_rate','target_fert']])
two_dim_pdp_plots(trained_rf_model, X_train = train_df[pred_cols],feat_combos=feat_combos)

In [ ]:
print(pred_cols)

trained_xgb_model, trained_xgb_model_features = fun.xgb_predict_spatial(
                input_df = train_df,
                pred_params = pred_cols,
                target_col = 'yield_mass_dry',
                block_size=mean_swath_width
)

In [ ]:
model_analysis.run_full_diagnostics(trained_xgb_model, X_train, y_train, X_test, y_test, pred_cols,'applied_rate','target_fert', metrics_only = True)

In [ ]:
model_analysis.run_full_diagnostics(trained_xgb_model, X_train, y_train, X_test, y_test, pred_cols,'applied_rate','target_fert')

In [ ]:
all_models_untuned = [trained_rf_model_features, trained_xgb_model_features]

# 2. Generate unified plots
fun.plot_comparative_parity(all_models_untuned)
fun.plot_comparative_residuals(all_models_untuned)
fun.plot_comparative_importances(all_models_untuned)

In [ ]:
print(pred_cols)


best_rf_model, rf_study = fun.tune_spatial_optuna(
    input_df=train_df,
    pred_params=pred_cols,
    target_col='yield_mass_dry',
    model_type='rf',
    n_trials=50,
    block_size=mean_swath_width
)

In [ ]:
rf_params_optuna = {'n_estimators': 450, 'max_depth': 50, 'min_samples_split': 4, 'max_features': 'log2', 'min_samples_leaf': 2}
#rf_params_optuna = rf_study.best_params
print(rf_params_optuna)

In [ ]:
print(pred_cols)


optuna_tune_trained_rf_model, optuna_rf_model_features = fun.rf_predict_spatial(
                input_df = train_df,
                pred_params = pred_cols,
                target_col = 'yield_mass_dry',
                block_size= mean_swath_width,
                model_name = 'Tuned RF model',
                model_params= rf_params_optuna
)

In [ ]:
model_analysis.run_full_diagnostics(optuna_tune_trained_rf_model, X_train, y_train, X_test, y_test, pred_cols,'applied_rate','target_fert', metrics_only = False)

In [ ]:
fun.plot_three_pdp_ice(optuna_tune_trained_rf_model, train_df[pred_cols], ['target_fert', 'elevation', 'slope_deg'])

In [ ]:
int_feats = ['elevation', 'target_fert','applied_rate','slope_deg','aspect_north','aspect_east']

In [ ]:
psa.shap_values(optuna_tune_trained_rf_model, test_df, pred_cols, yield_colname, sample_frac = 0.3, interactions = False, interaction_features= int_feats)

In [ ]:
psa.shap_values(optuna_tune_trained_rf_model, train_df, pred_cols, yield_colname, sample_frac = 0.2, interactions = False, interaction_features= int_feats)

In [ ]:
feat_combos = np.array([['applied_rate','target_fert'],
                        ['slope_deg','elevation'],
                        ['aspect_east', 'aspect_north'],
                        ['aspect_east','elevation'],
                        ['aspect_north','elevation'],
                        ['slope_deg','aspect_east'],
                        ['slope_deg','aspect_north'],
                        ['slope_deg','target_fert'],
                        ['elevation','target_fert'],
                        ['aspect_east','target_fert'],
                        ['aspect_north','target_fert']
                        ])

two_dim_pdp_plots(optuna_tune_trained_rf_model, X_train = train_df[pred_cols],feat_combos=feat_combos)

In [ ]:
# optuna tuning
print(pred_cols)

best_xgb_model, xgb_study = fun.tune_spatial_optuna(
    input_df=train_df,
    pred_params=pred_cols,
    target_col='yield_mass_dry',
    model_type='xgb',
    n_trials=30,      
    block_size=mean_swath_width
)


In [ ]:
#xgb_params_optuna = {'n_estimators': 450, 'learning_rate': 0.05, 'max_depth': 15, 'subsample': 0.5, 'colsample_bytree': 0.9}
xgb_params_optuna = xgb_study.best_params
print(xgb_params_optuna)

In [ ]:
print(pred_cols)


optuna_tune_trained_xgb_model, optuna_xgb_model_features = fun.xgb_predict_spatial(
                input_df = train_df,
                pred_params = pred_cols,
                target_col = 'yield_mass_dry',
                block_size= mean_swath_width,
                model_name = 'Tuned XGB model',
                model_params= xgb_params_optuna
)

In [ ]:
model_analysis.run_full_diagnostics(optuna_tune_trained_xgb_model, X_train, y_train, X_test, y_test, pred_cols,'applied_rate','target_fert', metrics_only = False)

In [ ]:
psa.shap_values(optuna_tune_trained_xgb_model, test_df, pred_cols, yield_colname,sample_frac = 0.2, interactions = True, interaction_features= int_feats)

In [ ]:
psa.shap_values(optuna_tune_trained_xgb_model, train_df, pred_cols, yield_colname, sample_frac = 0.2, interactions = True, interaction_features= int_feats)

In [ ]:
fun.plot_three_pdp_ice(optuna_tune_trained_xgb_model, train_df[pred_cols], ['target_fert', 'elevation', 'slope_deg'])

In [ ]:
feat_combos = np.array([['applied_rate','target_fert'],
                        ['slope_deg','elevation'],
                        ['aspect_east', 'aspect_north'],
                        ['aspect_east','elevation'],
                        ['aspect_north','elevation'],
                        ['slope_deg','aspect_east'],
                        ['slope_deg','aspect_north'],
                        ['slope_deg','target_fert'],
                        ['elevation','target_fert'],
                        ['aspect_east','target_fert'],
                        ['aspect_north','target_fert']
                        ])

two_dim_pdp_plots(optuna_tune_trained_xgb_model, X_train = train_df[pred_cols],feat_combos=feat_combos)

In [ ]:
compare_two_dim_pdp_plots(
    model_one   = optuna_tune_trained_rf_model,
    model_two   = optuna_tune_trained_xgb_model,
    model_names = ["RF", "XGB"],
    X_train     = train_df[pred_cols],
    feat_combos = np.array([['applied_rate','target_fert']])
)

In [ ]:

all_models = [optuna_rf_model_features, optuna_xgb_model_features]

fun.plot_comparative_parity(all_models)
fun.plot_comparative_residuals(all_models)
fun.plot_comparative_importances(all_models)

In [ ]:
rf_preds = optuna_tune_trained_rf_model.predict(imputed_grid_clean[pred_cols])

res = imputed_grid_clean[yield_colname] - rf_preds
results_df = gpd.GeoDataFrame({'true': imputed_grid_clean[yield_colname],
                               'predicted': rf_preds,
                               'residuals': res},
                              geometry=imputed_grid_clean.geometry,
                              crs=imputed_grid_clean.crs)

t = np.array(np.linspace(0.2,3.0,7).round(1))

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
results_df.plot(column='true', cmap='RdYlGn', legend=True, ax=ax[0], vmax = vmax, vmin = vmin,
                 legend_kwds={'label': 'Yield (t/ha)', 'ticks': t})
ax[0].set_title('True Yield')
ax[0].set_xlabel('Easting')
ax[0].set_ylabel('Northing')
ax[0].xaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
ax[0].yaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
results_df.plot(column='predicted', cmap='RdYlGn', legend=True, ax=ax[1], vmax = vmax, vmin = vmin,
                 legend_kwds={'label': 'Yield (t/ha)', 'ticks': t})
ax[1].set_title('Predicted Yield')
ax[1].set_xlabel('Easting')
ax[1].set_ylabel('Northing')
ax[1].xaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
ax[1].yaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
results_df.plot(column='residuals', cmap='coolwarm', legend=True, ax=ax[2], vmax = 1.5, vmin = -1.5,
                 legend_kwds={'label': 'Residuals (t/ha)', 'ticks': ticker.MaxNLocator(nbins=(5)).tick_values(-1.5, 1.5)})
ax[2].set_title('Residuals')
ax[2].set_xlabel('Easting')
ax[2].set_ylabel('Northing')
ax[2].xaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
ax[2].yaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))

plt.tight_layout()
plt.show()

In [ ]:
xgb_preds = optuna_tune_trained_xgb_model.predict(imputed_grid_clean[pred_cols])

res = imputed_grid_clean[yield_colname] - xgb_preds
results_df = gpd.GeoDataFrame({'true': imputed_grid_clean[yield_colname],
                               'predicted': xgb_preds,
                               'residuals': res},
                              geometry=imputed_grid_clean.geometry,
                              crs=imputed_grid_clean.crs)

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
results_df.plot(column='true', cmap='RdYlGn', legend=True, ax=ax[0], vmax = vmax, vmin = vmin,
                 legend_kwds={'label': 'Yield (t/ha)', 'ticks': t})
ax[0].set_title('True Yield')
ax[0].set_xlabel('Easting')
ax[0].set_ylabel('Northing')
ax[0].xaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
ax[0].yaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
results_df.plot(column='predicted', cmap='RdYlGn', legend=True, ax=ax[1], vmax = vmax, vmin = vmin,
                 legend_kwds={'label': 'Yield (t/ha)', 'ticks': t})
ax[1].set_title('Predicted Yield')
ax[1].set_xlabel('Easting')
ax[1].set_ylabel('Northing')
ax[1].xaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
ax[1].yaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
results_df.plot(column='residuals', cmap='coolwarm', legend=True, ax=ax[2], vmax = 1.5, vmin = -1.5,
                 legend_kwds={'label': 'Residuals (t/ha)', 'ticks': ticker.MaxNLocator(nbins=(5)).tick_values(-1.5, 1.5)})
ax[2].set_title('Residuals')
ax[2].set_xlabel('Easting')
ax[2].set_ylabel('Northing')
ax[2].xaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))
ax[2].yaxis.set_major_locator(ticker.MaxNLocator(nbins=(3)))

plt.tight_layout()
plt.show()

### GRF model implementation via pyGRF

In [ ]:
from spatial_random_forest_implementation import prep_dataset, spatial_parameters, train_grf_model, plot_results, plot_local_r2_error, plot_residuals, plot_local_feat_imp, plot_actual_pred_res, plot_parity_residuals

In [ ]:
X_train, X_test, y_train, y_test, training_coords, testing_coords = prep_dataset(imputed_grid_clean, yield_colname, pred_cols, NON_FEATURE_COLS)

In [ ]:
bandwidth, local_weight = spatial_parameters(y_train, X_train)

In [ ]:
pygrf_model, predict_combined, predict_global, local_importance_df, global_importance_df = train_grf_model(X_train, y_train, X_test, y_test, training_coords, testing_coords, bandwidth, local_weight)


In [ ]:
gdf_train, gdf_test = plot_results(y_test, predict_combined, testing_coords, training_coords, pygrf_model, pred_cols, local_importance_df):


In [ ]:
plot_local_r2_error(pygrf_model, training_coords)

In [ ]:
plot_residuals(gdf_test)

In [ ]:
feats = ['elevation',seed_colname]
plot_local_feat_imp(gdf_train, feats)

In [ ]:
feats = ['aspect_north','aspect_east']
plot_local_feat_imp(gdf_train, feats)

In [ ]:
feats = ['slope','elevation']
plot_local_feat_imp(gdf_train, feats)

In [ ]:
plot_actual_pred_res(predict_combined, testing_coords, y_test)

In [ ]:
plot_parity_residuals(y_test, predict_combined)